# Project 11 -- Niharika Rai

**TA Help:** (for instance) John Smith, Alice Jones, etc., list names of any TAs who helped you

- For example: Help with figuring out how to write a function (describe the tasks that they helped you with)

**Collaboration:** My Friend in CS, My Uncle, Another Student, etc., list names of any other people who helped you

(describe the tasks that they helped you with)
- For example: helped figuring out how to load the dataset.
- Another example: helped debug error with my plot.

**Internet Resources:** Stack Exchange, Stack Overflow, etc.

(describe any information that you learned from internet resources, including the URLs)
- data frames in Pandas versus R from StackOverflow  https://stackoverflow.com/questions/8991709/why-were-pandas-merges-in-python-faster-than-data-table-merges-in-r-in-2012

**ChatGPT, Gemini, Claude, etc:** Any language models or generative AI chatbots that helped you.

(if you used any such tools, please tell us here)
- For example:  I asked ChatGPT how to define a new data frames
- Another example:  Gemini told me how to make a function for sorting my data

- ***Link to AI Chat History***: Here is the link: https://chatgpt.com/share/6913a6b6-a9dc-800d-bd59-0a69a9fc0597. Please share a link to your chat if you used AI (ex. ChatGPT Shared Links)
**OVERALL MESSAGE:** Any time that you used anything except your brain to solve the questions in these projects, you need to disclose such resources at the start of the project, with details about your usage of the tools.

**YOUR OWN WORK:** Even when you utilize other resources, do NOT just copy and paste.  Write all explanations in your own words, using several sentences in English, which are understandable and which you wrote (and did not just copy and paste).

## Question 1

In [1]:
from pymongo import MongoClient
import pandas as pd
import json

# create a client to connect to the local MongoDB instance
client = MongoClient('mongodb://localhost:27017/')

# Create or connect to a database
db = client['ecommerce_db']

# Create or connect to collections
products = db['products']
customers = db['customers']
orders = db['orders']

In [2]:
# Load the e-commerce dataset from JSON file
with open('/anvil/projects/tdm/data/ecommerce/fake_ecommerce_dataset.json', 'r') as file:
    dataset = json.load(file)

# Insert the data into MongoDB
products.insert_many(dataset['products'])
customers.insert_many(dataset['customers'])
orders.insert_many(dataset['orders'])

print("Dataset loaded successfully!")

Dataset loaded successfully!


In [3]:
# Examine a sample product document to understand its structure

sample_product = products.find_one()
print("Sample Product Structure:")
print(f"Product Name: {sample_product['name']}")
print(f"Category: {sample_product['category']}")
print(f"Price: ${sample_product['price']}")
print(f"Specifications: {sample_product['specifications']}")
print(f"Number of Reviews: {len(sample_product['reviews'])}")
print(f"Tags: {sample_product['tags']}")

Sample Product Structure:
Product Name: Wireless Headphones
Category: Electronics
Price: $99.99
Specifications: {'brand': 'TechSound', 'color': 'Black', 'battery_life': '20 hours', 'connectivity': 'Bluetooth 5.0', 'weight': '250g', 'warranty': '2 years'}
Number of Reviews: 3
Tags: ['audio', 'wireless', 'bluetooth', 'headphones']


In [129]:
# Find products in a specific category
electronics = list(products.find({"category": "Electronics"}))
print(f"Electronics products: {len(electronics)}")

# Find products with price greater than $200
expensive_products = list(products.find({"price": {"$gt": 200}}))
print(f"Products over $200: {len(expensive_products)}")

# Find customers from a specific state
ca_customers = list(customers.find({"address.state": "CA"}))
print(f"Customers from California: {len(ca_customers)}")

# Find orders with specific status
completed_orders = list(orders.find({"status": "completed"}))
print(f"Completed orders: {len(completed_orders)}")

Electronics products: 6
Products over $200: 4
Customers from California: 3
Completed orders: 5


For all the parts in this question, I used the sample query given by the question to get all the outputs. Then, each of the basic queries first, find all the electronics, customers from California and orders that have been completed. The output that is displayed are the counts for each of these. From the output, we can see that there are 6 electronic products, 3 customers from California and 5 completed orders. Furthermore, the dot notation helps to query nested fields in MongoDB. For example, in 'address.state' dot notation, the query helps to access the state field which is inside the address object. Then, in 'specifications.brand' dot notation, the query helps to access the brand field which is inside the specifications object for further analysis.  

## Question 2

In [8]:
# Query arrays using $elemMatch
five_star_products = list(products.find({
    "reviews": {"$elemMatch": {"rating": 5}}
}))
print(f"Products with 5-star reviews: {len(five_star_products)}")

Products with 5-star reviews: 15


In [9]:
specific_user_products = list(products.find({
    "reviews": {"$elemMatch": {"user": "john_doe"}}
}))
print(f"Products with John Doe User: {len(specific_user_products)}")

Products with John Doe User: 1


In [10]:
high_rated_products = list(products.find({"reviews": {"$elemMatch": {"rating": {"$gte": 4}, "comment": {"$regex": "excellent", "$options": "i"}}}}))
print(f"Products with high rating and quality: {len(high_rated_products)}")

Products with high rating and quality: 3


For parts 2.1, 2.2 and 2.3, I used the sample queries from the question and modified them to get the required outputs. I used ChatGPT to debug my code for part 2.3 and referred to Piazza discussion. I also looked at project 10 code syntax when constructing the part "4 or above" in my query. Based on the outputs, I found that there are 15 products that have 5-star reviews, 1 product with John Doe as their user and 3 products that have a rating of 4 or above and contain the word "excellent" in their comment. To answer part 2.4, I would say the main difference between querying arrays with and without 'elemMatch' is that the 'elemMatch' operator allows you to query arrays where at least one element matches multiple criteria. Hence, when you do not use the 'elemMatch' operator, the conditions can be satisfied by different array elements. But when you do use the 'elemMatch' operator, all conditions must be met by the same array element. 

## Question 3

In [33]:
# Calculate average rating for each product using $unwind
pipeline = [
    {"$unwind": "$reviews"},  # Create one document per review
    {"$group": {
        "_id": "$product_id",
        "product_name": {"$first": "$name"},
        "avg_rating": {"$avg": "$reviews.rating"},
        "review_count": {"$sum": 1}
    }},
    {"$sort": {"avg_rating": -1}}
]

top_rated = list(products.aggregate(pipeline))
print("Average Rating for Each Product:")
for product in top_rated: 
    print(f"{product['product_name']}: {product['avg_rating']:.2f}")

Average Rating for Each Product:
Wireless Headphones: 4.67
Microwave: 4.67
Gaming Mouse: 4.67
Smart Watch: 4.67
Running Shoes: 4.67
Basketball: 4.67
Smartphone: 4.67
Laptop: 4.67
Air Fryer: 4.67
Tennis Racket: 4.67
Dumbbells Set: 4.33
Yoga Mat: 4.33
Tablet: 4.33
Blender: 4.33
Coffee Maker: 4.00


In [36]:
category_pipeline = [
    {"$group": {
        "_id": "$category",
        "product_name": {"$first": "$name"},
        "count": {"$sum": 1},
        "max_price": {"$max": "$price"},
        "total_stock": {"$sum": "$stock"}
    }},
    {"$sort": {"count": -1}}
]

category_stats = list(products.aggregate(category_pipeline))
print("Most Expensive Products by category:")
for category in category_stats:
    print(f"{category['_id']}: {category['product_name']}, {category['count']} products, max price: ${category['max_price']:.2f}")

Most Expensive Products by category:
Electronics: Wireless Headphones, 6 products, max price: $1299.99
Sports: Running Shoes, 5 products, max price: $199.99
Kitchen: Coffee Maker, 4 products, max price: $159.99


In [27]:
# Find customers count by state
state_pipeline = [
    {"$group": {
        "_id": "$address.state",
        "customer_count": {"$sum": 1}}
        },
    {"$sort": {"customer_count": -1}}
]

state_stats = list(customers.aggregate(state_pipeline))
print("\nCustomers by state:")
for state in state_stats:
    print(f"{state['_id']}: {state['customer_count']} customers.")


Customers by state:
TX: 4 customers.
CA: 3 customers.
AZ: 1 customers.
WA: 1 customers.
PA: 1 customers.
NY: 1 customers.
FL: 1 customers.
NC: 1 customers.
IL: 1 customers.
OH: 1 customers.


In [62]:
# Calculate total revenue by product category from orders using $lookup
revenue_pipeline = [
    {"$unwind": "$items"},  # Create one document per order item
    {"$lookup": {
        "from": "products",
        "localField": "items.product_id",
        "foreignField": "product_id",
        "as": "product_info"
    }},
    {"$unwind": "$product_info"},  # Unwind the joined product data
    {"$group": {
        "_id": "$product_info.category",
        "revenue": {"$sum": {"$multiply": ["$items.price", "$items.quantity"]}}
    }},
    {"$sort": {"revenue": -1}}
]

revenue_by_category = list(orders.aggregate(revenue_pipeline))
print("Revenue by category:")
for category in revenue_by_category:
    print(f"{category['_id']}: ${category['revenue']:.2f}")

Revenue by category:
Electronics: $2979.93
Sports: $1074.90
Kitchen: $819.92


In [59]:
pipeline = [
    {"$group": {"_id": "$customer_id", "order_count": {"$sum": 1}}},
    {"$match": {"order_count": {"$gte": 1}}},
    {"$lookup": {
        "from": "customers",
        "localField": "_id",
        "foreignField": "customer_id",
        "as": "customer_info"
    }},
    {"$unwind": "$customer_info"},
    {"$sort": {"order_count": -1}}
]

results = list(orders.aggregate(pipeline))
for r in results:
    print(f"{r['customer_info']['name']} ({r['_id']}) has {r['order_count']} order")

Lisa Davis (C006) has 1 order
Robert Garcia (C007) has 1 order
Mike Johnson (C003) has 1 order
John Doe (C001) has 1 order
Emily Martinez (C008) has 1 order
Sarah Wilson (C004) has 1 order
James Anderson (C009) has 1 order
Jane Smith (C002) has 1 order
Maria Rodriguez (C010) has 1 order
David Brown (C005) has 1 order


In [60]:
pipeline = [
    {"$group": {"_id": "$customer_id", "order_count": {"$sum": 1}}},
    {"$match": {"order_count": {"$gt": 1}}},
    {"$lookup": {
        "from": "customers",
        "localField": "_id",
        "foreignField": "customer_id",
        "as": "customer_info"
    }},
    {"$unwind": "$customer_info"},
    {"$sort": {"order_count": -1}}
]

results = list(orders.aggregate(pipeline))
for r in results:
    print(f"{r['customer_info']['name']} ({r['_id']}) has {r['order_count']} orders")

For parts 3.1, 3.2 and 3.3, 3.4 and 3.5, I used the sample queries from the question, modified them and used ChatGPT to debug my code. I was really confused with part 3.5 but later found out that there are no customers who have placed more than 1 order. Hence, even the query runs, there is no output produced. However, there are many customers who have placed just 1 order and I have printed their names along with their customer ids to show this. To answer part 3.6, I would say that the '$unwind' operator takes an array field and creates one output document for each element in the array, which further allows the user to perform many different types of calculations on the array element values. It is useful because it is much more effective than the traditional SQL GROUP BY operators and it allows you to work with complex nested data structures.

## Question 4

In [63]:
import pandas as pd
from pymongo import MongoClient

# Convert MongoDB collection to DataFrame
def mongo_to_dataframe(collection, query=None):
    if query is None:
        query = {}
    cursor = collection.find(query)
    return pd.DataFrame(list(cursor))

# Get products data
products_df = mongo_to_dataframe(products)
print(f"Products DataFrame shape: {products_df.shape}")

Products DataFrame shape: (15, 10)


In [65]:
# Analyze product data with pandas
products_df = mongo_to_dataframe(products)
print("Product Analysis:")
print(f"Total products: {len(products_df)}")
print(f"Average price: ${products_df['price'].mean():.2f}")
print(f"Price range: ${products_df['price'].min():.2f} - ${products_df['price'].max():.2f}")

Product Analysis:
Total products: 15
Average price: $257.66
Price range: $24.99 - $1299.99


In [66]:
# Category analysis
category_analysis = products_df.groupby('category').agg({
    'price': ['mean', 'count'],
    'stock': 'sum'
}).round(2)
print("\nCategory Analysis:")
print(category_analysis)


Category Analysis:
              price       stock
               mean count   sum
category                       
Electronics  479.99     6   158
Kitchen      112.49     4    77
Sports       106.99     5   137


In [68]:
# Work with customer data and nested addresses
customers_df = mongo_to_dataframe(customers)
print(f"Total customers: {len(customers_df)}")

Total customers: 15


In [72]:
# Extract state information from nested address
customers_df['state'] = customers_df['address'].apply(lambda x: x['state'])
state_analysis = customers_df['state'].value_counts()
print("\nCustomers by State:")
print(state_analysis)


Customers by State:
state
TX    4
CA    3
NY    1
IL    1
AZ    1
PA    1
FL    1
OH    1
NC    1
WA    1
Name: count, dtype: int64


In [84]:
customers_df['preferences']

0     {'categories': ['Electronics', 'Sports'], 'new...
1     {'categories': ['Kitchen', 'Electronics'], 'ne...
2     {'categories': ['Sports', 'Electronics'], 'new...
3     {'categories': ['Kitchen', 'Sports'], 'newslet...
4     {'categories': ['Electronics'], 'newsletter': ...
5     {'categories': ['Kitchen', 'Sports', 'Electron...
6     {'categories': ['Sports'], 'newsletter': False...
7     {'categories': ['Electronics', 'Kitchen'], 'ne...
8     {'categories': ['Sports', 'Electronics'], 'new...
9     {'categories': ['Kitchen'], 'newsletter': Fals...
10    {'categories': ['Electronics', 'Sports'], 'new...
11    {'categories': ['Kitchen', 'Sports'], 'newslet...
12    {'categories': ['Electronics'], 'newsletter': ...
13    {'categories': ['Sports', 'Kitchen', 'Electron...
14    {'categories': ['Electronics', 'Sports'], 'new...
Name: preferences, dtype: object

In [75]:
# Analyze newsletter subscriptions
newsletter_subscribers = customers_df['preferences'].apply(lambda x: x['newsletter']).sum()
print(f"\nNewsletter subscribers: {newsletter_subscribers}")
print(f"Subscription rate: {newsletter_subscribers/len(customers_df)*100:.1f}%")


Newsletter subscribers: 10
Subscription rate: 66.7%


In [80]:
notification_enabled = customers_df['preferences'].apply(lambda x: x['notifications']).sum()
print(f"\nCustomers with notifications enabled: {notification_enabled}")
print(f"Rate: {notification_enabled/len(customers_df)*100:.1f}%")


Customers with notifications enabled: 10
Rate: 66.7%


In [83]:
language_counts = customers_df['preferences'].apply(lambda x: x['language']).value_counts()
print("\nCustomer language preferences:")
print(language_counts)


Customer language preferences:
preferences
English    13
Spanish     2
Name: count, dtype: int64


In [85]:
from collections import Counter

all_categories = customers_df['preferences'].apply(lambda x: x['categories']).explode()
category_counts = all_categories.value_counts()
print("\nFavorite product categories among customers:")
print(category_counts)


Favorite product categories among customers:
preferences
Electronics    11
Sports         10
Kitchen         7
Name: count, dtype: int64


In [86]:
# Calculate inventory value by category
products_df['inventory_value'] = products_df['price'] * products_df['stock']
inventory_by_category = products_df.groupby('category')['inventory_value'].sum().sort_values(ascending=False)
print("\nInventory Value by Category:")
print(inventory_by_category)


Inventory Value by Category:
category
Electronics    45898.42
Sports         10023.63
Kitchen         8199.23
Name: inventory_value, dtype: float64


In [115]:
orders_df

,_id,order_id,customer_id,order_date,status,total_amount,shipping_address,items,payment_method,shipping_cost,tax,month,month_name
0,69136a32e8e6c2cf0e07c7d7,O001,C001,2024-01-15 10:30:00+00:00,completed,229.98,"{'street': '123 Main St', 'city': 'New York', ...","[{'product_id': 'P001', 'quantity': 1, 'price'...",credit_card,9.99,18.00,1,January
1,69136a32e8e6c2cf0e07c7d8,O002,C002,2024-01-20 14:15:00+00:00,completed,159.98,"{'street': '456 Oak Ave', 'city': 'Los Angeles...","[{'product_id': 'P002', 'quantity': 1, 'price'...",paypal,9.99,12.00,1,January
2,69136a32e8e6c2cf0e07c7d9,O003,C003,2024-01-25 09:45:00+00:00,shipped,179.98,"{'street': '789 Pine St', 'city': 'Chicago', '...","[{'product_id': 'P005', 'quantity': 2, 'price'...",credit_card,9.99,13.50,1,January
3,69136a32e8e6c2cf0e07c7da,O004,C004,2024-02-01 16:20:00+00:00,processing,209.98,"{'street': '321 Elm St', 'city': 'Houston', 's...","[{'product_id': 'P009', 'quantity': 1, 'price'...",credit_card,9.99,15.75,2,February
4,69136a32e8e6c2cf0e07c7db,O005,C005,2024-02-05 11:30:00+00:00,completed,1099.98,"{'street': '654 Maple Ave', 'city': 'Phoenix',...","[{'product_id': 'P004', 'quantity': 1, 'price'...",credit_card,19.99,82.50,2,February
5,69136a32e8e6c2cf0e07c7dc,O006,C006,2024-02-08 13:45:00+00:00,shipped,249.98,"{'street': '987 Cedar Rd', 'city': 'Philadelph...","[{'product_id': 'P006', 'quantity': 1, 'price'...",paypal,19.99,109.50,2,February
6,69136a32e8e6c2cf0e07c7dd,O007,C007,2024-02-10 08:15:00+00:00,completed,174.98,"{'street': '147 Birch Ln', 'city': 'San Antoni...","[{'product_id': 'P008', 'quantity': 1, 'price'...",credit_card,9.99,13.13,2,February
7,69136a32e8e6c2cf0e07c7de,O008,C008,2024-02-12 15:30:00+00:00,processing,479.98,"{'street': '258 Spruce St', 'city': 'San Diego...","[{'product_id': 'P001', 'quantity': 1, 'price'...",credit_card,9.99,36.00,2,February
8,69136a32e8e6c2cf0e07c7df,O009,C009,2024-02-14 12:00:00+00:00,shipped,429.98,"{'street': '369 Willow Way', 'city': 'Dallas',...","[{'product_id': 'P003', 'quantity': 1, 'price'...",paypal,9.99,32.25,2,February
9,69136a32e8e6c2cf0e07c7e0,O010,C010,2024-02-16 10:45:00+00:00,completed,269.98,"{'street': '741 Poplar Dr', 'city': 'San Jose'...","[{'product_id': 'P002', 'quantity': 1, 'price'...",credit_card,9.99,20.25,2,February


In [128]:
# Convert the order_date column to datetime
orders_df['order_date'] = pd.to_datetime(orders_df['order_date'])

# month number (1-12)
orders_df['month'] = orders_df['order_date'].dt.month

# month name for better readability
orders_df['month_name'] = orders_df['order_date'].dt.strftime('%B')

orders_by_month = orders_df.groupby('month_name').size()
print(orders_by_month)

month_name
February    7
January     3
dtype: int64


In [117]:
completed_orders = orders_df[orders_df['status'] == 'completed']
completed_by_month = completed_orders.groupby('month_name').size()
print(completed_by_month)

month_name
February    3
January     2
dtype: int64


In [118]:
status_by_month = orders_df.groupby(['month_name', 'status']).size().unstack(fill_value=0)
print(status_by_month)

status      completed  processing  shipped
month_name                                
February            3           2        2
January             2           0        1


In [123]:
# Explode order items array into separate rows for analysis
orders_df = mongo_to_dataframe(orders)
items_df = pd.DataFrame([item for row in orders_df["items"] for item in row])
print(f"Total order items: {len(items_df)}")

Total order items: 24


In [124]:
# Join with product information to get categories
products_df = mongo_to_dataframe(products)
items_df = items_df.merge(
    products_df[["product_id", "category"]],
    on="product_id",
    how="left"
)

In [125]:
# Analyze items by category
category_analysis = items_df.groupby("category").agg({
    "quantity": "sum",
    "price": "mean"
}).round(2).sort_values(by='quantity', ascending=False)
print("\nOrder items by category:")
print(category_analysis)


Order items by category:
             quantity   price
category                     
Sports             10  116.10
Kitchen             8  102.49
Electronics         7  425.70


I used the sample queries in the question to get my outputs. But, I also used ChatGPT to debug my code, understand the questions better when I was confused and get the required outputs. For part 4.2, since preferences were not specified, I used the sample query in the question for newsletter subscriptions and went further to analysis all the other preferences given in the preferences column of the customers_df. For part 4.4, I was confused by what order patterns specifically wanted me to do, so I computed all the order patterns possible. For this, I computed all the orders by month, only completed orders by month and the orders sorted by their status for each month. There were only 2 months in the dataset and I converted them to both the month number and names for better readability. To answer part 4.6, I would say that the benefits of using MongoDB with Python data analysis libraries are that you get the combined benefits of both MongoDB and Python data analysis which makes analyzing data more flexible and easier. For example, MongoDB can handle complex, nested data structures while Python libraries provide advanced statistical and analytical capabilities. Then, Pandas makes data preprocessing easier and Matplotlib and Seaborn make data visualization more effective. Hence, the main benefit is to be able to do complex calculations and find meaningful insights. 

## Question 5

In [130]:
# Create indexes for better performance
products.create_index("category")
products.create_index("price")
products.create_index([("category", 1), ("price", -1)])  # Compound index

'category_1_price_-1'

In [131]:
# Test query performance
query = {"category": "Electronics"}
explain_result = db.command("explain", {"find": "products", "filter": query})
stats = explain_result.get("executionStats", {})
print("\nQuery Performance (category = Electronics):")
print("Execution time (ms):", stats.get("executionTimeMillis", "N/A"))
print("Documents examined:", stats.get("totalDocsExamined", "N/A"))
print("Documents returned:", stats.get("nReturned", stats.get("totalDocsReturned", "N/A")))


Query Performance (category = Electronics):
Execution time (ms): 1
Documents examined: 6
Documents returned: 6


In [132]:
# Create additional specialized indexes
products.create_index("tags")  # Multikey index for arrays
products.create_index("specifications.brand")  # Index on nested field
customers.create_index("email", unique=True)  # Unique index
orders.create_index([("customer_id", 1), ("order_date", -1)])  # Compound index
products.create_index([("name", "text")])  # Text index for product names
print("\nCreated specialized indexes (tags, brand, email(unique), compound, text).")


Created specialized indexes (tags, brand, email(unique), compound, text).


In [133]:
# Test different query patterns
queries_to_test = [
    {"category": "Electronics"},
    {"tags": "wireless"},
    {"specifications.brand": "Samsung"},
    {"price": {"$gt": 200, "$lt": 500}},
]

print("\nQuery Performance Analysis:")
for i, query in enumerate(queries_to_test, 1):
    result = db.command("explain", {"find": "products", "filter": query})
    stats = result.get("executionStats", {})
    print(f"Query {i}: {query}")
    print("  Execution time (ms):", stats.get("executionTimeMillis", "N/A"))
    print("  Docs examined:", stats.get("totalDocsExamined", "N/A"))
    print("  Docs returned:", stats.get("nReturned", stats.get("totalDocsReturned", "N/A")))
    print()


Query Performance Analysis:
Query 1: {'category': 'Electronics'}
  Execution time (ms): 0
  Docs examined: 6
  Docs returned: 6

Query 2: {'tags': 'wireless'}
  Execution time (ms): 0
  Docs examined: 1
  Docs returned: 1

Query 3: {'specifications.brand': 'Samsung'}
  Execution time (ms): 0
  Docs examined: 0
  Docs returned: 0

Query 4: {'price': {'$gt': 200, '$lt': 500}}
  Execution time (ms): 0
  Docs examined: 2
  Docs returned: 2



In [134]:
products_df.columns

Index(['_id', 'product_id', 'name', 'category', 'price', 'stock',
       'specifications', 'reviews', 'tags', 'created_at'],
      dtype='object')

In [137]:
products.create_index("reviews.rating")  # Index on nested field

'reviews.rating_1'

In [139]:
products.create_index("reviews.comment") 

'reviews.comment_1'

In [141]:
products.find({"reviews.rating": {"$gte": 4}}).explain()

{'explainVersion': '1',
 'queryPlanner': {'namespace': 'ecommerce_db.products',
  'indexFilterSet': False,
  'parsedQuery': {'reviews.rating': {'$gte': 4}},
  'queryHash': 'D39BD826',
  'planCacheKey': '8EB7A9FC',
  'optimizationTimeMillis': 0,
  'maxIndexedOrSolutionsReached': False,
  'maxIndexedAndSolutionsReached': False,
  'maxScansToExplodeReached': False,
  'winningPlan': {'stage': 'FETCH',
   'inputStage': {'stage': 'IXSCAN',
    'keyPattern': {'reviews.rating': 1},
    'indexName': 'reviews.rating_1',
    'isMultiKey': True,
    'multiKeyPaths': {'reviews.rating': ['reviews']},
    'isUnique': False,
    'isSparse': False,
    'isPartial': False,
    'indexVersion': 2,
    'direction': 'forward',
    'indexBounds': {'reviews.rating': ['[4, inf.0]']}}},
  'rejectedPlans': []},
 'executionStats': {'executionSuccess': True,
  'nReturned': 15,
  'executionTimeMillis': 1,
  'totalKeysExamined': 30,
  'totalDocsExamined': 15,
  'executionStages': {'stage': 'FETCH',
   'nReturned': 1

In [148]:
my_queries_to_test = [
    {"reviews.rating": {"$gte": 4}},
    {"reviews.comment": {"$regex": "excellent", "$options": "i"}}
]

print("\nQuery Performance Analysis:")
for i, query in enumerate(my_queries_to_test, 1):
    result = db.command("explain", {"find": "products", "filter": query})
    stats = result.get("executionStats", {})
    print(f"Query {i}: {query}")
    print("  Execution time (ms):", stats.get("executionTimeMillis", "N/A"))
    print("  Docs examined:", stats.get("totalDocsExamined", "N/A"))
    print("  Docs returned:", stats.get("nReturned", stats.get("totalDocsReturned", "N/A")))
    print()


Query Performance Analysis:
Query 1: {'reviews.rating': {'$gte': 4}}
  Execution time (ms): 0
  Docs examined: 15
  Docs returned: 15

Query 2: {'reviews.comment': {'$regex': 'excellent', '$options': 'i'}}
  Execution time (ms): 0
  Docs examined: 15
  Docs returned: 3



In [153]:
# Check index usage statistics
db_stats = db.command("collStats", "products")
print("Collection Statistics:")
print(f"Total documents: {db_stats.get('count', 'N/A')}")
print(f"Average document size: {db_stats.get('avgObjSize', 0):.2f} bytes")
print(f"Total collection size: {db_stats.get('size', 0)/1024:.2f} KB")
print(f"Total index size: {db_stats.get('totalIndexSize', 0)/1024:.2f} KB")

# List all indexes
indexes = list(products.list_indexes())
print(f"\nIndexes on 'products' collection: {len(indexes)}")
for idx in indexes:
    print(f"- {idx['name']}: {idx['key']}")

# Test aggregation performance
agg_pipeline = [
    {"$group": {"_id": "$category", "count": {"$sum": 1}}},
    {"$sort": {"count": -1}}
]
agg_explain = db.command("explain", {"aggregate": "products", "pipeline": agg_pipeline, "cursor": {}}, verbosity="executionStats")
print(f"\nAggregation execution time: {agg_explain.get('executionStats', {}).get('executionTimeMillis', 'N/A')}ms")

Collection Statistics:
Total documents: 15
Average document size: 714.00 bytes
Total collection size: 10.46 KB
Total index size: 180.00 KB

Indexes on 'products' collection: 9
- _id_: SON([('_id', 1)])
- category_1: SON([('category', 1)])
- price_1: SON([('price', 1)])
- category_1_price_-1: SON([('category', 1), ('price', -1)])
- tags_1: SON([('tags', 1)])
- specifications.brand_1: SON([('specifications.brand', 1)])
- name_text: SON([('_fts', 'text'), ('_ftsx', 1)])
- reviews.rating_1: SON([('reviews.rating', 1)])
- reviews.comment_1: SON([('reviews.comment', 1)])

Aggregation execution time: N/Ams


In [154]:
exec_ms = None

# 1) Some versions place it top-level:
exec_ms = exec_ms or agg_explain.get("executionStats", {}).get("executionTimeMillis")

# 2) Aggregation often returns per-stage stats with estimates — scan for them:
def _find_time(d):
    if isinstance(d, dict):
        if "executionTimeMillis" in d:
            return d["executionTimeMillis"]
        if "executionTimeMillisEstimate" in d:
            return d["executionTimeMillisEstimate"]
        for v in d.values():
            t = _find_time(v)
            if t is not None:
                return t
    elif isinstance(d, list):
        for x in d:
            t = _find_time(x)
            if t is not None:
                return t
    return None

# If returned in stages, use our scan function:
exec_ms = exec_ms or _find_time(agg_explain.get("stages", []))
print("Aggregation execution time (ms):", exec_ms)

Aggregation execution time (ms): 0


For all parts, I mostly used the sample queries in the question. However, I also used ChatGPT to debug my code and test out the performance for the 'reviews.rating' nested field. For part 5.5, since no example of complex aggregation pipelines was specified, I used the sample queries in the question. I also executed the 'scanning' of the execution time as I was curious about its output. To answer part 5.6, I would say that the key considerations for optimizing MongoDB performance in production are considering index strategy, ensuring sufficient RAM for the working set, using explain() to identify slow queries, setting up monitoring and alerts for performance metrics, implementing regular backups and point-in-time recovery, using authentication and authorization, and planning for horizontal scaling with sharding. I would also say that the main reason for tracking all these considerations is to ensure that results are returned quickly and speed is optimized.  

## Pledge

By submitting this work I hereby pledge that this is my own, personal work. I've acknowledged in the designated place at the top of this file all sources that I used to complete said work, including but not limited to: online resources, books, and electronic communications. I've noted all collaboration with fellow students and/or TA's. I did not copy or plagiarize another's work.

> As a Boilermaker pursuing academic excellence, I pledge to be honest and true in all that I do. Accountable together – We are Purdue.

https://www.purdue.edu/odos/osrr/honor-pledge/
